<div align="right"><a href="L33-Yule-Walker-und-Autokorrelationsmethode.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). Please note that for running MATLAB in Notebooks, you have to install the [MATLAB API for Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) first. The installation process is described [here](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/S33-Yule-Walker-und-Autokorrelationsmethode.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S32 Power Spectral Density Estimation using the Bartlett-Welch Method](./S32-PSD-Estimation-using-the-Bartlett-Welch-Method.ipynb) | [Exercise 33](./E33-Yule-Walker-und-Autokorrelationsmethode.ipynb) | [Contents / Overview](./00-MATLAB-Notebooks-Overview-EN.ipynb) | Next: [S34 Burg-Algorithm](./S34-Burg-Algorithm.ipynb)

</div>

---


### Exercise: Yule–Walker

The goal of this exercise is to analyze spectrum estimation by applying the Yule–Walker resp. the autocorrelation approach. It is crucial for this procedure that the process $X(k)$, which is to be analyzed, is set to zero outside the measured interval $0 \leq k \leq N-1$. Thus the *transient behavior* of the predictor is taken into account to calculate the prediction error (eq. (9.3.7)). In spite of the influence of the transient behavior, this approach is denoted to be stationary. The reason for this is the calculation of the coefficients of the correlation matrix, which has the form of a Hermitian Toeplitz matrix. Such a matrix occurs as an autocorrelation matrix of stationary processes too.

In addition, the biased autocorrelation series $\hat{r}_{XX}(\kappa)$ is used. Therefore, the estimation of the AR-coefficients is biased as well.

To do our analysis we use an (all-pole) system $H(z)=1/A(z)$ with

$$
A(z)=1 + 0.3611z^{-1} - 0.1373z^{-2} + 0.2685z^{-3} - 0.0533z^{-4} - 0.3353z^{-5} - 0.307z^{-6}
$$

of order $m=6$, which is excited by a stationary, zero mean, white process $N(k)$ with power $\sigma_N^2 = 1$.

<img src="./pngs/l15sys.png" width="70%">

---

#### a)

Plot the impulse response $h(k)$ of the system using command `impz(·)` within the interval $0 \leq k \leq 80$, and determine the pole-zero plane using command `zplane(·)`.

How many samples are necessary for the system to tune in, that is $h(k) \approx 0$?

---

#### b)

Determine the exact autocorrelation series $r_{XX}(\kappa)$ using the impulse response for $0 \leq \kappa \leq 6$. Calculate the exact coefficients $a_0,\ldots,a_n$ of the AR($n$)-model using the Yule–Walker equation (9.3.17):

$$
\frac{1}{A_n(z)} = \frac{1}{a_0 + a_1 z^{-1} + \cdots + a_n z^{-n}},
\qquad \text{with} \quad n=6
$$

(For this special case $n$ is set to $6$, whereby the order of the AR-model equals the order of the system $H(z)$.)

To calculate, use the command `lyw(·,·,1)`.

Furthermore, calculate the exact power density spectrum $S_{XX}(\mathrm{e}^{j\Omega})$ and plot your results.

---

#### c)

Following, the influence of the length $N$ of the samples of the stochastic process $X(k)$ is analyzed for the estimation of the power density spectrum. For this, $1000$ realizations of samples of the process $X(k)$ are created, whereby one has to consider that the system has to be tuned within the interval of $N$ samples, that is $h(k) \approx 0$ (command: `filter(·)`).

The measurement interval should include $N=10$ resp. $N=256$ values. Calculation should be done using the MATLAB function `lyw(·,·,1)`.

At first, determine the estimated coefficients $\hat{a}*{6,1},\ldots,\hat{a}*{6,6}$ of the AR(6)-model as well as the mean value and the variance for each coefficient using all $1000$ ensembles. Plot the results.

To analyze the bias, determine the power of the deviation between the mean values and their exact values of the AR-coefficients:

$$
\frac{1}{n}\sum_{i=0}^{n} \left| \bar{a}_i - a_i \right|^2
$$

whereby $\bar{a}_i$ is the mean value and $a_i$ the exact coefficient.

Based on the estimated coefficients, calculate the power density spectrum for each ensemble. Determine the mean power density spectrum as well as the variance of the estimation. Plot the exact and the mean spectrum as well as the upper $2\sigma$-borders (95%-borders) in a single figure.

In [ ]:
# BEGIN SOLUTION
MA = [1];
AR = [1.0000 0.3611 -0.1373 0.2685 -0.0533 -0.3353 -0.307];
NFFT = 2^10;
n_a = length(AR);
n = n_a-1;          % Filter order

% #####  Part a:  Transient Behavior #####
% I.A./exact ACF (steady state)
[h_arma,t] = impz(MA,AR,80);

% # Graphical Output # %
figure;
stem([1:length(h_arma)],h_arma(1:length(h_arma)));
grid;
xlabel('k');
ylabel('h(k)');
title(sprintf('Impulse Response of the AR(%d) Model',n));

figure;
[h1,h2,h3]=zplane(MA,AR);
set(h1,'Visible','off');
title(sprintf('Poles of the AR(%d) Model',n));
grid;
xlabel('Real Part');
ylabel('Imaginary Part');
disp('Number of samples required for the transient: N_vor=80');

input('....Press RETURN to continue:');

% #####  Part b:  Exact ACF r_xx(0)...r_xx(n), YW Coefficients, PSD #####
rxx        = xcorr(h_arma);
index      = (length(rxx)+1)/2;
rxx_ex     = rxx(index:index+n);
ARcoeff    = lyw(h_arma,n,0);
Omega_norm = 0:1/NFFT:1-1/NFFT;
Sxx_ex     = abs(fft(MA,NFFT)./fft(AR,NFFT)).^2;

% # Graphical Output # %
figure;
stem([0:n],rxx_ex,'r-');
grid;
xlabel('\kappa');
ylabel('r_{xx}(\kappa)');
title(sprintf('Exact ACF Values for AR(%d): r_{xx}(0)...r_{xx}(%d)',n,n));

figure;
stem([0:n],ARcoeff,'r');
axis([0 n 1.1*min(ARcoeff) 1.1*max(ARcoeff)]);
xlabel('i');
ylabel('a_i');
grid;
title(sprintf('Exact AR(%d) Coefficients: a_0...a_{%d}',n,n));

figure;
plot(Omega_norm, 10*log10(Sxx_ex/max(Sxx_ex)),'r--');
grid;
xlabel('\Omega/2\pi');
ylabel('S_{xx}(e^{j\Omega}) in dB');
title('Exact PSD');

input('....Press RETURN to continue:');

disp('....Please wait: Calculation in progress.....');

% #####  Part c:  Influence of the Length N of the Data Sequence x(k) #####
N = [10 256];
N_vor = 80;
mc_run = 1e3;

for run0 = N
    Sxx_ar = zeros(mc_run,NFFT);
    Coeff = zeros(mc_run,n_a);
    
    for run1 = 1:mc_run
        noise = randn(1,run0+N_vor);
        x_all = filter(MA,AR,noise);
        x = x_all(N_vor+1:length(x_all));
        ARcoeff = lyw(x,n,1);
        Coeff(run1,:) = ARcoeff;
        Sxx_ar(run1,:) = abs(ones(1,NFFT)./fft(ARcoeff,NFFT)).^2;
    end;
    
    Coeff_mean = mean(Coeff,1);
    Coeff_std = std(Coeff,1);
    C2sig_o = Coeff_mean+2*Coeff_std;
    C2sig_u = Coeff_mean-2*Coeff_std;
    Coeff_lst = mean(abs(AR-Coeff_mean).^2);
    Sxx_mean = mean(Sxx_ar,1);
    Sxx_2sigma = Sxx_mean+2*std(Sxx_ar,1);

    % # Graphical Output # %
    figure;
    h1 = stem([0:n],AR,'ro');               hold on;
    h2 = stem([0:n],Coeff_mean,'bx');       hold on;
    h3 = stem([0:n],C2sig_o,'go','filled'); hold on;
    h4 = stem([0:n],C2sig_u,'go','filled');
    legend([h1(1),h2(1),h3(1)],'Exact',['Mean./E\{e^2\}=',sprintf('%1.2e',Coeff_lst)],...
        ['2\sigma-Range/ \sigma=',sprintf('%1.2f',mean(Coeff_std))]);
    set(h1,'Linestyle','none');
    set(h2,'Linestyle','none');
    set(h3,'Linestyle','none');
    set(h4,'Linestyle','none');
    x_lines = [[0:n]; [0:n]; nan*ones(1,n+1)];
    stddev_lines = [C2sig_u; C2sig_o; nan*ones(1,n+1)];
    plot(x_lines(:),stddev_lines(:),'g');
    title(sprintf('AR(%d) Coefficients; N=%d',n,run0));
    xlabel('i');
    ylabel('Coefficients');
    grid;
    axis([0 n -1.5 1.5]);
    hold off;

    figure;
    h1 = plot(Omega_norm, 10*log10(Sxx_ex/max(Sxx_ex)),'r--');       hold on;
    h2 = plot(Omega_norm, 10*log10(Sxx_mean/max(Sxx_ex)),'b-');      hold on;
    h3 = plot(Omega_norm, 10*log10(Sxx_2sigma/max(Sxx_ex)),'g-.');   hold off;
    ylabel('S_{xx}(e^{j\Omega}) in dB');
    legend([h1,h2,h3],'Exact','Mean','2\sigma-Range');
    title(sprintf('PSD from YW-AR(%d); N=%d',n, run0));
    xlabel('\Omega/2\pi');
    grid;
end;
% #####  EOF  #####
# END SOLUTION

### Solution: Yule–Walker

* **a)**
  The following figures show the impulse response $h(k)$ of the AR(6)-system in the interval
  $0 \leq k \leq 80$. It can be stated that the underlying system for time values
  $k \geq 60$ has reached its steady state.

  Additionally, the position of the poles
  $z_{\infty,1}=0.3444 + i\cdot 0.8315$,
  $z_{\infty,2}=z_{\infty,1}^{*}$,
  $z_{\infty,3}=0.85$,
  $z_{\infty,4}=-0.91$,
  $z_{\infty,5}=-0.49 + i\cdot 0.4950$,
  $z_{\infty,6}=z_{\infty,5}^{*}$
  are depicted in the pole-zero diagram.

  <img src="./pngs/ldsv9a02_a1.png" width="45%">
  <img src="./pngs/ldsv9a02_a2.png" width="45%">

---

* **b)**
  The relation
  $r_{NN}(\kappa)=\sigma_N^2\delta(\kappa) ;\Leftrightarrow;
  S_{NN}(\mathrm{e}^{j\Omega})=\sigma_N^2$
  applies for stationary, zero mean, white noise. At the system output, the true ACF resp. the
  exact spectral power density exist with $\sigma_N^2=1$:

  $$
  r_{XX}(\kappa)
  = r_{HH}^{(E)}(\kappa) \ast r_{NN}(\kappa)
  = \sigma_N^2 \cdot r_{hh}^{(E)}(\kappa)
  = r_{HH}^{(E)}(\kappa)
  := \sum_{k=-\infty}^{\infty} h^*(k),h(k+\kappa)
  $$

  $$
  S_{XX}(\mathrm{e}^{j\Omega})
  = |H(\mathrm{e}^{j\Omega})|^2 \cdot S_{NN}(\mathrm{e}^{j\Omega})
  = \sigma_N^2 \cdot |H(\mathrm{e}^{j\Omega})|^2
  = |H(\mathrm{e}^{j\Omega})|^2
  $$

  For the calculation of the AR-coefficients using the Yule–Walker equation (9.3.17), the
  autocorrelation series is considered at the positions
  $r_{XX}(0), \ldots, r_{XX}(n)$. It can be remarked that the parameter $n$
  describes the order of the AR($n$)-process and does not have to satisfy the
  order $m$ of the underlying system $H(z)$ (underestimation resp. overestimation
  of the number of poles).

  Besides the description of the exact ACF-series, the exact AR(6)-coefficients
  $a_0,\ldots,a_6$ are shown in the following figure. The calculation occurs by
  using the Levinson–Durbin recursion for solving the Yule–Walker equation (9.3.17).

  For the power density spectra (PDS)
  $S_{XX}(\mathrm{e}^{j\Omega}) \leftrightarrow r_{XX}(\kappa)$, the trend is plotted
  as a dashed line in the bottom figure. The calculation of the power density spectra
  is done by using the relationship

  $$
  S_{XX}(\mathrm{e}^{j\Omega})
  = \sigma_N^2 \cdot |H(\mathrm{e}^{j\Omega})|^2
  = \left| \frac{B(\mathrm{e}^{j\Omega})}{A(\mathrm{e}^{j\Omega})} \right|^2
  = \frac{\left| \mathcal{F}{b(k)} \right|^2}{\left| \mathcal{F}{a(k)} \right|^2}.
  $$

  For a graphical representation of the true power density spectrum
  $S_{XX}(\mathrm{e}^{j\Omega})$, the discrete-time Fourier transform
  $\mathcal{F}{\cdot}$ would be approximated by a $2^{10}$-point FFT.
  Since the transmission system $H(z)$ is real-valued, a symmetric power density
  spectrum normalized by the frequency $\Omega/2\pi$ is obtained.

  <img src="./pngs/ldsv9a02_b1.png" width="30%">
  <img src="./pngs/ldsv9a02_b2.png" width="30%">
  <img src="./pngs/ldsv9a02_b3.png" width="30%">

---

* **c)**
  The left-hand figure shows the results of the coefficient estimation of an AR(6)-model
  by generating $1000$ sample functions with different lengths $N=10$ resp. $N=256$
  of the output process $X(k)$. It is assumed that the system $H(z)$ has reached its
  steady state (the setting times are not included in the sampled series).

  For the case $N=10$, the bias of the autocorrelation method is clearly identified;
  the zero mean (crosses) strongly deviates from the exact coefficients (circles).
  This behavior is described through the average (mean) power error $E{e^2}$, which
  serves as a measure to evaluate the bias and in this case has a value not equal to zero.
  Furthermore, a relatively large estimation variance exists for a small number of
  samples ($N=10$), represented by the standard deviation $\sigma$.

  A comparison of the coefficient estimations for different measurement interval lengths
  $N=10$ resp. $N=256$ shows an explicit improvement with increasing $N$. The deviation
  between the mean and the exact value is significantly smaller for $N=256$, but still
  not equal to zero, as seen for example in $\hat{a}_{6,6}$. The variance resp. standard
  deviation of the estimation also decreases.

  The right figure compares the exact power density spectrum with the estimated mean
  power density spectrum. To describe the estimated variance, the upper $2\sigma$ limits
  are also plotted. The results clearly reflect the influence of the length of the
  observation interval on the estimation quality. For the considered system $H(z)$,
  a relatively good approximation of the exact power density spectrum using the
  autocorrelation method is achieved for $N=256$. However, it must be noted that the
  estimation is principally biased.

  <img src="./pngs/ldsv9a02_c1.png" width="22%">
  <img src="./pngs/ldsv9a02_c2.png" width="22%">
  <img src="./pngs/ldsv9a02_c3.png" width="22%">
  <img src="./pngs/ldsv9a02_c4.png" width="22%">